<!-- 학습 보강 셀 -->

# 12. Loading Saved ChromaDB 학습 흐름

이 노트북은 11번에서 저장한 ChromaDB 컬렉션을 다시 불러와 질의하는 예제입니다.
핵심은 원본 문서를 다시 로드하거나 다시 임베딩하지 않고, 저장된 벡터 DB를 재사용한다는 점입니다.

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [ ]:
# LLM과 임베딩 모델 설정
# - 저장할 때 사용한 임베딩 모델과 같은 모델을 사용해야 검색 품질이 유지됩니다.
llm = Ollama(
    model='gemma2:2b',
    temperature=0.5,
    request_timeout=120,
)

embed_model = OllamaEmbedding(
    model_name='nomic-embed-text',
)

In [ ]:
# 저장된 ChromaDB 열기
# - 11번 노트북에서 만든 ./chroma_db 디렉토리가 있어야 합니다.
db = chromadb.PersistentClient(
    path='./chroma_db',
)

# 기존 컬렉션 로드
# - get_or_create_collection을 사용하면 컬렉션이 없을 때 빈 컬렉션이 만들어집니다.
# - 실습 편의상 사용하지만, 실제 운영에서는 get_collection으로 존재 여부를 엄격히 확인하는 편이 안전합니다.
chroma_collection = db.get_or_create_collection('quickstart_ollama')

<!-- 학습 보강 셀 -->

## 이 노트북을 실행하기 전 조건

`./chroma_db` 폴더와 `quickstart_ollama` 컬렉션은 11번 노트북을 실행해야 만들어집니다.
폴더가 없거나 컬렉션이 비어 있으면 인덱스 객체는 만들어져도 검색 결과가 없을 수 있습니다.

In [ ]:
# ChromaDB 컬렉션을 LlamaIndex vector_store로 감쌉니다.
vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection,
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# 저장된 벡터로부터 인덱스 로드
# - from_vector_store는 기존 Chroma 컬렉션을 검색 가능한 VectorStoreIndex로 감쌉니다.
# - 이 단계는 원본 PDF를 다시 읽거나 다시 임베딩하지 않습니다.
index = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
    embed_model=embed_model,
)

<!-- 학습 보강 셀 -->

## from_vector_store의 의미

`VectorStoreIndex.from_vector_store()`는 이미 존재하는 벡터 저장소를 LlamaIndex 쿼리 엔진에서 사용할 수 있게 감싸는 단계입니다.
새 임베딩을 만드는 단계가 아니라, 저장된 벡터를 검색 가능한 인터페이스로 연결하는 단계입니다.

In [ ]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

In [ ]:
# 쿼리 실행
query = '이 논문에서 제안하는 모델의 장점은 무엇인가? 한글로 답변해줘'
response = query_engine.query(query)

print()
print('질문:', query)
print('답변:', response)

<!-- 학습 보강 셀 -->

## 로드 성공 여부를 판단하는 기준

답변이 생성되는 것만으로는 충분하지 않습니다.
질문과 관련된 문서가 실제 ChromaDB에서 검색되었는지 확인하려면 필요할 때 `response.source_nodes`를 출력해 파일명과 page_label을 함께 확인하세요.